In [1]:
import numpy as np
import cv2
import random
import rasterio
import os

In [2]:
def ensure_three_channels(image):
    """
    Ensures the image has 3 channels (RGB). If the image has 1 channel, it converts it to 3 channels.
    """
    if len(image.shape) == 2 or image.shape[2] == 1:  # Grayscale or single channel
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    return image

In [3]:
def add_random_noise(image):
    """
    Adds random noise to the image.
    """
    noise_type = random.choice(['gaussian', 'speckle'])
    if noise_type == 'gaussian':
        mean = 0
        std_dev = random.uniform(0.07, 0.2) * np.amax(image)
        noise = np.random.normal(mean, std_dev, image.shape)
        noisy_image = image + noise
    elif noise_type == 'speckle':
        noise = np.random.randn(*image.shape) * 0.25  # Reduced speckle intensity
        noisy_image = image + image * noise
    return np.clip(noisy_image, 0, 1)

In [4]:
def add_blur(image):
    """
    Adds random blur to the image.
    """
    image = (image * 255.0).astype(np.uint8)
    blur_type = random.choice(['gaussian', 'median'])
    if blur_type == 'gaussian':
        ksize = random.choice([3, 5])  # Kernel size
        blurred_image = cv2.GaussianBlur(image, (ksize, ksize), 0)
    elif blur_type == 'median':
        ksize = random.choice([3, 5])  # Kernel size must be odd
        blurred_image = cv2.medianBlur(image, ksize)
    return blurred_image.astype(np.float32) / 255.0  # Normalize back to [0, 1]

In [5]:
def add_blocky_effects(image):
    """
    Adds blocky compression-like effects to the image.
    """
    block_size = 2
    h, w, _ = image.shape
    blocky_image = image.copy()
    for i in range(0, h, block_size):
        for j in range(0, w, block_size):
            block = blocky_image[i:i+block_size, j:j+block_size]
            avg_color = np.mean(block, axis=(0, 1))
            blocky_image[i:i+block_size, j:j+block_size] = avg_color
    return blocky_image

In [6]:
def degrade_image(image):
    """
    Randomly applies at least one operation (noise, blur, or blocky effects) to the image.
    """
    operations = [add_random_noise, add_blur, add_blocky_effects]
    random.shuffle(operations)

    # Always apply at least one operation
    degraded_image = operations[0](image)

    # Apply the remaining operations with 50% probability
    for operation in operations[1:]:
        if random.random() < 0.5:  # 50% chance for additional operations
            degraded_image = operation(degraded_image)

    return degraded_image

In [13]:
def process_and_resize_image(image, output_128_lr, output_degraded_path_128, output_128, output_512, train):
    """
    Resizes the image to (256, 256), ensures 3 channels, saves the original as JPG, and saves the degraded version as JPG.
    """
    # Resize the original image
    resized_image = cv2.resize(image, (512, 512), interpolation=cv2.INTER_AREA)
    resized_image = ensure_three_channels(resized_image)  # Ensure 3 channels
    
    # Save the original resized image as JPG
    if train:
        cv2.imwrite(output_512, (resized_image * 255).astype(np.uint8))
    
    image_128 = cv2.resize(image, (128, 128), interpolation=cv2.INTER_AREA)
    image_128 = ensure_three_channels(image_128)  # Ensure 3 channels
    if train:
        cv2.imwrite(output_128_lr, (image_128 * 255).astype(np.uint8))
    cv2.imwrite(output_128, (image_128 * 255).astype(np.uint8))
    
    # Degrade the image
    degraded_image = degrade_image(image_128)
    degraded_image = (degraded_image * 255).astype(np.uint8)
    
    # Save the degraded image as JPG
    cv2.imwrite(output_degraded_path_128, degraded_image)

In [14]:
def process_tiff(input_path, output_128_lr, output_degraded_path_128, output_128, output_512, train):
    """
    Reads a GeoTIFF file using rasterio, resizes it, ensures 3 channels, saves the original as JPG, and saves the degraded image as JPG.
    """
    with rasterio.open(input_path) as dataset:
        image = dataset.read([1, 2, 3]) if dataset.count >= 3 else dataset.read(1)  # Read bands
        image = np.moveaxis(image, 0, -1)  # Move channel axis to the end
        image = image.astype(np.float32)
        image = (image - np.min(image)) / (np.max(image) - np.min(image))  # Normalize to [0, 1]
        image = ensure_three_channels(image)
    
    process_and_resize_image(image, output_128_lr, output_degraded_path_128, output_128, output_512, train)

In [15]:
def process_standard_image(input_path, output_128_lr, output_degraded_path_128, output_128, output_512, train):
    """
    Reads a standard image file, resizes it, ensures 3 channels, saves the original as JPG, and saves the degraded image as JPG.
    """
    image = cv2.imread(input_path, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError("Could not open the input file.")
    
    image = image.astype(np.float32) / 255.0  # Normalize to [0, 1]
    process_and_resize_image(image, output_128_lr, output_degraded_path_128, output_128, output_512, train)

In [16]:
def process_image(input_path, output_128_lr, output_degraded_path_128, output_128, output_512, train = True):
    """
    Determines the type of input file and processes it accordingly.
    """
    _, ext = os.path.splitext(input_path)
    if ext.lower() in ['.tif', '.tiff']:
        process_tiff(input_path, output_128_lr, output_degraded_path_128, output_128, output_512, train)
    elif ext.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
        process_standard_image(input_path, output_128_lr, output_degraded_path_128, output_128, output_512, train)
    else:
        raise ValueError("Unsupported file format. Supported formats: TIFF, JPG, PNG, BMP.")


In [17]:
dataset1_path = os.listdir('Dataset1/train/')
dataset2_path = os.listdir('Dataset2/train/')
dataset3_path = os.listdir('Dataset3/train/')

In [12]:
for i in range(1):
    for image in dataset1_path:
        process_image('Dataset1/train/' + image, 'ESRGAN_DS/LR/' + image, 'CycleGAN_DS/trainA/' + image, 'CycleGAN_DS/trainB/' + image, 'ESRGAN_DS/HR/' + image)
    for image in dataset2_path:
        if 'mask' not in image:
            process_image('Dataset2/train/' + image, 'ESRGAN_DS/LR/' + image, 'CycleGAN_DS/trainA/' + image, 'CycleGAN_DS/trainB/' + image, 'ESRGAN_DS/HR/' + image)
    for directory in dataset3_path:
        for image in os.listdir('Dataset3/train/' + directory + '/images/'):
            process_image('Dataset3/train/' + directory + '/images/' + image, 'ESRGAN_DS/LR/' + image, 'CycleGAN_DS/trainA/' + image, 'CycleGAN_DS/trainB/' + image, 'ESRGAN_DS/HR/' + image)

In [19]:
for i in range(1):
    for image in os.listdir('Dataset1/test'):
        process_image('Dataset1/test/' + image, '', 'CycleGAN_DS/testA/' + image, 'CycleGAN_DS/testB/' + image, '', train = False)
    for image in os.listdir('Dataset2/test'):
        if 'mask' not in image:
            process_image('Dataset2/test/' + image, '', 'CycleGAN_DS/testA/' + image, 'CycleGAN_DS/testB/' + image, '', train = False)